# Grid Search pesado — Random Forest y LightGBM

Búsqueda refinada para una ejecución larga. Random Forest evalúa 54 combinaciones y LightGBM 81, usando 5 folds por sujeto: 675 fits en total, más dos reajustes finales.

Cada fit se ejecuta por separado y cada modelo utiliza todos los núcleos disponibles. Con una referencia conservadora de 40 segundos por fit, los 675 fits representan unas 7.5 horas, más los reajustes finales. Conviene reservar varias horas porque las configuraciones con más árboles y features pueden tardar más.

MLflow está activado y registrará una corrida por búsqueda con el grid, el mejor resultado CV, los mejores parámetros, las métricas de validación y el mejor modelo.

## 1. Configuración intensiva

La búsqueda usa un solo proceso para mostrar los logs en orden. Dentro de cada fit, Random Forest y LightGBM usan todos los núcleos disponibles mediante `n_jobs=-1`.

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'data.dvc').is_file())
PACKAGE_SRC = REPO_ROOT / 'packages' / 'sleep-staging' / 'src'
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

from IPython.display import display
from sklearn.model_selection import ParameterGrid
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import discover_sleep_telemetry_records
from sleep_staging.training import (
    MlflowConfig, compare_evaluations,
    create_lightgbm_classifier, create_random_forest_classifier,
    display_evaluation, grouped_grid_search,
    load_or_build_supervised_dataset, split_by_subject,
)

DATA_DIR = REPO_ROOT / 'data' / 'sleep-telemetry'
DATASET_CACHE_DIR = REPO_ROOT / 'data' / 'processed' / 'supervised'
SEED = 42
VALIDATION_SIZE = 0.20
CV_SPLITS = 5
SEARCH_N_JOBS = 1
MAX_RECORDS = None
MLFLOW = MlflowConfig(
    enabled=True,
    experiment_name='sleep_staging_grid_search_heavy',
    tracking_uri='sqlite:///mlflow.db',
    log_model=True,
)
PIPELINE = PreprocessingPipeline(PreprocessingConfig())
print(f'Procesadores disponibles: {os.cpu_count()} | procesos de búsqueda: {SEARCH_N_JOBS}')

## 2. Dataset completo y separación por sujeto

La búsqueda usa únicamente training. El holdout se evalúa una sola vez después de seleccionar la mejor configuración.

In [ ]:
records = discover_sleep_telemetry_records(DATA_DIR)
records = records if MAX_RECORDS is None else records[:MAX_RECORDS]
dataset = load_or_build_supervised_dataset(records, PIPELINE, DATASET_CACHE_DIR)
split = split_by_subject(dataset, validation_size=VALIDATION_SIZE, random_state=SEED)

print(f'Dataset: {dataset.features.shape[0]} épocas x {dataset.features.shape[1]} features')
print(f'Training: {split.X_train.shape} | sujetos: {len(split.train_subjects)}')
print(f'Validación: {split.X_validation.shape} | sujetos: {len(split.validation_subjects)}')
assert set(split.train_subjects).isdisjoint(split.validation_subjects)
display(dataset.labels.value_counts().rename('épocas').sort_index().to_frame())

## 3. Grids refinados a partir de la búsqueda ligera

En la búsqueda ligera, Random Forest obtuvo su mejor resultado con `max_depth=20`, `max_features='sqrt'`, `min_samples_leaf=2` y `n_estimators=500` (`F1_macro CV=0.7344`). LightGBM ganó con `learning_rate=0.03`, `n_estimators=500`, `num_leaves=24` y `min_child_samples=40` (`F1_macro CV=0.7574`). Ambas combinaciones están incluidas exactamente en los grids siguientes.

En LightGBM la distancia entre el mejor y el peor candidato ligero fue apenas 0.00165, muy inferior a la desviación entre folds (aproximadamente 0.010–0.012). Por eso no se interpreta el primer puesto como una diferencia concluyente: el grid pesado explora una vecindad local alrededor de 500 árboles, tasa 0.03, 24 hojas y 40 muestras por hoja. En Random Forest se concentra la profundidad alrededor de 20 y se conservan las dos opciones de `max_features` que ocuparon los primeros lugares.

In [ ]:
RF_GRID = {
    'n_estimators': [300, 500, 700],
    'max_depth': [15, 20, 25],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 0.1],
}
LGBM_GRID = {
    'n_estimators': [400, 500, 600],
    'learning_rate': [0.025, 0.03, 0.04],
    'num_leaves': [20, 24, 31],
    'min_child_samples': [30, 40, 50],
}

rf_candidates = len(list(ParameterGrid(RF_GRID)))
lgbm_candidates = len(list(ParameterGrid(LGBM_GRID)))
total_fits = (rf_candidates + lgbm_candidates) * CV_SPLITS
reference_minutes = total_fits * 40 / 60
print(f'Random Forest: {rf_candidates} combinaciones × {CV_SPLITS} = {rf_candidates * CV_SPLITS} fits')
print(f'LightGBM: {lgbm_candidates} combinaciones × {CV_SPLITS} = {lgbm_candidates * CV_SPLITS} fits')
print(f'Total: {total_fits} fits | referencia a 40 s/fit: {reference_minutes:.1f} minutos')

## 4. Grid Search pesado — Random Forest

In [ ]:
random_forest = create_random_forest_classifier(random_state=SEED, n_jobs=-1)
rf_search = grouped_grid_search(
    random_forest, RF_GRID, split, model_name='Random Forest optimizado heavy',
    n_splits=CV_SPLITS, n_jobs=SEARCH_N_JOBS, verbose=10,
    mlflow_config=MLFLOW,
)
print(f'Mejor F1 macro CV: {rf_search.best_cv_score:.4f}')
print(f'Mejores parámetros: {rf_search.best_parameters}')
display(rf_search.cv_results.head(20))

## 5. Grid Search pesado — LightGBM

In [ ]:
lightgbm = create_lightgbm_classifier(random_state=SEED, n_jobs=-1)
lgbm_search = grouped_grid_search(
    lightgbm, LGBM_GRID, split, model_name='LightGBM optimizado heavy',
    n_splits=CV_SPLITS, n_jobs=SEARCH_N_JOBS, verbose=10,
    mlflow_config=MLFLOW,
)
print(f'Mejor F1 macro CV: {lgbm_search.best_cv_score:.4f}')
print(f'Mejores parámetros: {lgbm_search.best_parameters}')
display(lgbm_search.cv_results.head(20))

## 6. Comparación final y MLflow

Para abrir la interfaz desde la raíz del proyecto después del entrenamiento: `mlflow ui --backend-store-uri sqlite:///mlflow.db`.

In [ ]:
print('MEJOR RANDOM FOREST')
print(rf_search.best_parameters)
print('\nMEJOR LIGHTGBM')
print(lgbm_search.best_parameters)

display(compare_evaluations(
    rf_search.validation_evaluation,
    lgbm_search.validation_evaluation,
).round(4))
display_evaluation(rf_search.validation_evaluation)
display_evaluation(lgbm_search.validation_evaluation)